In [0]:
import requests
import json
from datetime import datetime

# Retrieve secrets securely from Databricks
client_id = dbutils.secrets.get(scope="amazon", key="client_id")
client_secret = dbutils.secrets.get(scope="amazon", key="client_secret")
refresh_token = dbutils.secrets.get(scope="amazon", key="refresh_token")
auth_url = dbutils.secrets.get(scope="amazon", key="auth_url")

# Prepare the payload
payload = {
    'grant_type': 'refresh_token',
    'client_id': client_id,
    'client_secret': client_secret,
    'refresh_token': refresh_token
}

# Get access token from Amazon API
auth_response = requests.post(auth_url, data=payload)
auth_response.raise_for_status()

# Extract and display token
access_token = auth_response.json()['access_token']
print("🔐 LWA Access Token:", access_token[:5] + "...")


In [0]:
api_url = dbutils.secrets.get(scope="amazon", key="api_url")
params = {
    'MarketplaceIds': 'ATVPDKIKX0DER',
    'CreatedAfter': 'TEST_CASE_200'
}
headers = {
    'x-amz-access-token': access_token
}


response = requests.get(api_url, headers=headers, params=params)
print(response.status_code)
print(json.dumps(response.json()['payload']['Orders'], indent=2))
data = response.json()['payload']['Orders']

In [0]:
container_name = dbutils.widgets.get("container_name")
storage_account_name = dbutils.widgets.get("storage_account_name")
folder_name = dbutils.widgets.get("folder_name")
storage_account_key = dbutils.secrets.get(scope="amazon", key="storage_account_key")
folder_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/{folder_name}"
spark.conf.set(f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net", storage_account_key)
dbutils.fs.put(f"{folder_path}/{datetime.now()}.json", json.dumps(data), overwrite=True)